# 07. Verify A7+06 blend `model.ts` on DLC validation

목적: DLC validation 104개 전체에서 제출용 TorchScript의 `P(RERECORDED)`와

`p_expected = (1/3) * p_A7 + (2/3) * p_06`

를 sample-level로 비교한다.

중요 조건:
- 제출 `inference.py`와 동일한 FP32 / no autocast
- 16 frames / stride 2 / center clip
- 384 resize + center crop
- ImageNet normalization
- threshold = 0.5

확인 항목:
- probability absolute difference
- threshold 0.5 label flip
- DLC Macro-F1 변화
- 차이가 큰 sample
- 결과 CSV / JSON 저장

In [ ]:
# 1. Colab / Drive setup
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import time

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

cv2.setNumThreads(1)
torch.set_float32_matmul_precision('high')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'CUDA GPU에서 실행하세요.'

print('device:', DEVICE)
print('gpu   :', torch.cuda.get_device_name(0))
print('torch :', torch.__version__)

In [ ]:
# 2. Paths / knobs
PROJECT_DRIVE = Path('/content/drive/MyDrive/Blackbox-Detection')

BLEND_DIR = PROJECT_DRIVE / 'outputs/stage1/a7_06_probblend_2to1/vjepa2_1_b_blend'
MODEL_TS = BLEND_DIR / 'model.ts'

# 기존 A7 / 06 validation prediction CSV
A7_PRED_CSV = PROJECT_DRIVE / 'outputs/stage1/dlc/vjepa2_1_b/val_predictions.csv'
ADAPTED_06_PRED_CSV = PROJECT_DRIVE / 'outputs/stage1/dlc_ccd_synrr_200/vjepa2_1_b/val_predictions.csv'

# DLC data root 후보. 자동으로 못 찾으면 OVERRIDE만 수정.
DLC_ROOT_CANDIDATES = [
    PROJECT_DRIVE / 'data/stage1/dlc2021',
    PROJECT_DRIVE / 'datasets/stage1/dlc2021',
    PROJECT_DRIVE / 'datasets/dlc2021',
    Path('/content/Blackbox-Detection/data/stage1/dlc2021'),
    Path('/content/data/stage1/dlc2021'),
]
DLC_ROOT_OVERRIDE = None
# 예: DLC_ROOT_OVERRIDE = Path('/content/drive/MyDrive/Blackbox-Detection/data/stage1/dlc2021')

ALPHA_06 = 2.0 / 3.0
THRESHOLD = 0.5
VIDEO_EXT = {'.mp4','.avi','.mov','.mkv','.m4v','.3gp','.3gpp','.wmv','.webm'}

def first_existing(paths):
    for p in paths:
        if p.exists(): return p
    return None

DLC_ROOT = Path(DLC_ROOT_OVERRIDE) if DLC_ROOT_OVERRIDE is not None else first_existing(DLC_ROOT_CANDIDATES)

print('MODEL_TS :', MODEL_TS, '| exists:', MODEL_TS.is_file())
print('A7 CSV   :', A7_PRED_CSV, '| exists:', A7_PRED_CSV.is_file())
print('06 CSV   :', ADAPTED_06_PRED_CSV, '| exists:', ADAPTED_06_PRED_CSV.is_file())
print('DLC ROOT :', DLC_ROOT)

assert MODEL_TS.is_file(), f'model.ts not found: {MODEL_TS}'
assert A7_PRED_CSV.is_file(), f'A7 prediction CSV not found: {A7_PRED_CSV}'
assert ADAPTED_06_PRED_CSV.is_file(), f'06 prediction CSV not found: {ADAPTED_06_PRED_CSV}'
assert DLC_ROOT is not None and DLC_ROOT.is_dir(), 'DLC_ROOT_OVERRIDE를 실제 DLC 경로로 지정하세요.'

## CSV sanity check
A7/06 CSV를 `video_id` 기준으로 합쳐 expected probability blend를 만든다.
104개가 정확히 merge되지 않으면 중단한다.

In [ ]:
# 3. Load A7 / 06 prediction tables and build expected blend
a7 = pd.read_csv(A7_PRED_CSV)
m06 = pd.read_csv(ADAPTED_06_PRED_CSV)

required = {'video_id','label','prob_rerecorded'}
for name, frame in [('A7', a7), ('06', m06)]:
    missing = required - set(frame.columns)
    assert not missing, f'{name} CSV missing columns: {sorted(missing)}'
    assert frame['video_id'].is_unique, f'{name} video_id is not unique'

wide = a7[['video_id','label','prob_rerecorded']].rename(columns={'prob_rerecorded':'p_a7'}).merge(
    m06[['video_id','label','prob_rerecorded']].rename(columns={'label':'label_06','prob_rerecorded':'p_06'}),
    on='video_id', how='inner', validate='one_to_one'
)
assert (wide['label'] == wide['label_06']).all(), 'A7/06 labels disagree'
wide = wide.drop(columns='label_06').copy()
wide['p_expected'] = (1.0 - ALPHA_06) * wide['p_a7'] + ALPHA_06 * wide['p_06']
wide['expected_pred'] = np.where(wide['p_expected'] >= THRESHOLD, 'RERECORDED', 'ORIGINAL')

print('A7 rows       :', len(a7))
print('06 rows       :', len(m06))
print('merged rows   :', len(wide))
print('class balance :', wide['label'].value_counts().to_dict())
assert len(wide) == 104, f'Expected 104 DLC validation videos, got {len(wide)}'
display(wide.head())

In [ ]:
# 4. Macro-F1 helper
LABELS = ('ORIGINAL','RERECORDED')
def macro_f1(y_true, y_pred):
    y_true=np.asarray(y_true); y_pred=np.asarray(y_pred); scores=[]
    for cls in LABELS:
        tp=np.sum((y_true==cls)&(y_pred==cls))
        fp=np.sum((y_true!=cls)&(y_pred==cls))
        fn=np.sum((y_true==cls)&(y_pred!=cls))
        denom=2*tp+fp+fn
        scores.append(0.0 if denom==0 else (2.0*tp)/denom)
    return float(np.mean(scores))

for prob_col,name in [('p_a7','A7'),('p_06','06'),('p_expected','expected blend')]:
    pred=np.where(wide[prob_col].to_numpy()>=THRESHOLD,'RERECORDED','ORIGINAL')
    print(f'{name:14s} Macro-F1 @0.5 = {macro_f1(wide["label"],pred):.12f}')

## Resolve DLC paths
`dlc__aze_passport__00.or0001` 같은 `video_id`를 실제 파일로 resolve한다.
모든 104개가 1:1로 resolve되지 않으면 추론을 시작하지 않는다.

In [ ]:
# 5. Resolve every CSV video_id to one DLC file
all_videos=sorted(p for p in DLC_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in VIDEO_EXT)
print('video files found:',len(all_videos))
by_stem={}
for p in all_videos: by_stem.setdefault(p.stem.lower(),[]).append(p)

def resolve_video(video_id):
    parts=str(video_id).split('__'); stem=parts[-1].lower(); hints=[x.lower() for x in parts[1:-1]]
    candidates=by_stem.get(stem,[])
    if len(candidates)==1: return candidates[0]
    scored=[]
    for p in candidates:
        try: rel=p.relative_to(DLC_ROOT).as_posix().lower()
        except ValueError: rel=p.as_posix().lower()
        scored.append((sum(h in rel for h in hints),p))
    if scored:
        top=max(s for s,_ in scored)
        best=[p for s,p in scored if s==top and s>0]
        if len(best)==1: return best[0]
    raise RuntimeError(f'Could not uniquely resolve {video_id!r}; candidates={[str(p) for p in candidates[:10]]}')

resolved=[]; errors=[]
for vid in wide['video_id']:
    try: resolved.append(resolve_video(vid))
    except Exception as e: errors.append((vid,str(e))); resolved.append(None)
if errors:
    display(pd.DataFrame(errors,columns=['video_id','error']))
    raise RuntimeError('DLC path resolution failed')
wide['video_path']=[str(p) for p in resolved]
assert wide['video_path'].nunique()==len(wide)
print('[PASS] all 104 DLC videos resolved')
display(wide[['video_id','video_path']].head())

## Submission-equivalent preprocessing
제출 `inference.py`와 동일하게 FP32 / no autocast로 실행한다.

In [ ]:
# 6. Exact submission-equivalent preprocessing
S1_NUM_FRAMES=16; S1_STRIDE=2; S1_CROP_SIZE=384
S1_SHORT_SIDE=int(round(S1_CROP_SIZE*256.0/224.0))
S1_MEAN=torch.tensor([0.485,0.456,0.406],dtype=torch.float32)[:,None,None,None]
S1_STD=torch.tensor([0.229,0.224,0.225],dtype=torch.float32)[:,None,None,None]

def total_frames(path):
    cap=cv2.VideoCapture(str(path))
    try:
        if not cap.isOpened(): raise RuntimeError(f'cannot open {path}')
        total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    finally: cap.release()
    return max(total,1)

def center_clip_indices(total):
    total=max(int(total),1); span=(S1_NUM_FRAMES-1)*S1_STRIDE+1; max_start=max(total-span,0); start=max_start//2
    return np.clip(start+np.arange(S1_NUM_FRAMES,dtype=np.int64)*S1_STRIDE,0,total-1)

def decode_frames(path,frame_indices):
    wanted=[int(x) for x in np.asarray(frame_indices).reshape(-1)]
    order=np.argsort(wanted,kind='stable'); sorted_wanted=[wanted[i] for i in order]
    cap=cv2.VideoCapture(str(path))
    try:
        if not cap.isOpened(): raise RuntimeError(f'cannot open {path}')
        decoded=[None]*len(sorted_wanted); cap.set(cv2.CAP_PROP_POS_FRAMES,sorted_wanted[0]); position=sorted_wanted[0]; last=None
        for slot,target in enumerate(sorted_wanted):
            if last is not None and target<position: decoded[slot]=last; continue
            frame=None; ok=False
            while position<=target:
                ok,frame=cap.read(); position+=1
                if not ok: break
            if not ok or frame is None: decoded[slot]=last; continue
            last=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB); decoded[slot]=last
        if all(x is None for x in decoded): raise RuntimeError(f'cannot decode {path}')
        first=next(x for x in decoded if x is not None); filled=[x if x is not None else first for x in decoded]
    finally: cap.release()
    restored=[None]*len(wanted)
    for slot,orig in enumerate(order): restored[int(orig)]=filled[slot]
    return np.stack(restored)

def resize_shorter_side(frame,target):
    h,w=frame.shape[:2]
    if min(h,w)==target: return frame
    scale=target/float(min(h,w)); nw=max(target,int(round(w*scale))); nh=max(target,int(round(h*scale)))
    interp=cv2.INTER_AREA if scale<1.0 else cv2.INTER_LINEAR
    return cv2.resize(frame,(nw,nh),interpolation=interp)

def preprocess_video(path):
    frames=decode_frames(path,center_clip_indices(total_frames(path)))
    processed=np.empty((S1_NUM_FRAMES,S1_CROP_SIZE,S1_CROP_SIZE,3),dtype=np.uint8)
    for i,frame in enumerate(frames):
        resized=resize_shorter_side(frame,S1_SHORT_SIDE); h,w=resized.shape[:2]
        top=max((h-S1_CROP_SIZE)//2,0); left=max((w-S1_CROP_SIZE)//2,0)
        processed[i]=resized[top:top+S1_CROP_SIZE,left:left+S1_CROP_SIZE]
    clip=torch.from_numpy(np.ascontiguousarray(processed)).float().div_(255.0).permute(3,0,1,2).contiguous()
    clip=(clip-S1_MEAN)/S1_STD
    return clip.unsqueeze(0)

In [ ]:
# 7. Load model.ts and warm up with one real clip
model_ts=torch.jit.load(str(MODEL_TS),map_location=DEVICE).eval()
sample_path=Path(wide.iloc[0]['video_path'])
sample_clip=preprocess_video(sample_path).to(device=DEVICE,dtype=torch.float32)
with torch.inference_mode(): sample_logits=model_ts(sample_clip)
if isinstance(sample_logits,(tuple,list)): sample_logits=sample_logits[0]
print('sample output shape:',tuple(sample_logits.shape)); print('sample logits:',sample_logits.detach().cpu())
assert tuple(sample_logits.shape)==(1,2); assert torch.isfinite(sample_logits).all()
del sample_clip,sample_logits; torch.cuda.empty_cache()
print('[PASS] model.ts warmup')

## Run all 104 videos
의도적으로 batch=1, FP32, no autocast로 실행한다.

In [ ]:
# 8. model.ts inference on all 104 DLC videos
p_ts=[]; elapsed_each=[]; failures=[]
torch.cuda.synchronize(); total_started=time.perf_counter()
with torch.inference_mode():
    for row in tqdm(wide.itertuples(index=False),total=len(wide),desc='model.ts DLC'):
        path=Path(row.video_path)
        try:
            started=time.perf_counter()
            clip=preprocess_video(path).to(device=DEVICE,dtype=torch.float32,non_blocking=True)
            logits=model_ts(clip)
            if isinstance(logits,(tuple,list)): logits=logits[0]
            assert tuple(logits.shape)==(1,2); assert torch.isfinite(logits).all()
            prob=float(torch.softmax(logits.float(),dim=1)[0,1].cpu().item())
            torch.cuda.synchronize(); elapsed=time.perf_counter()-started
            p_ts.append(prob); elapsed_each.append(elapsed); del clip,logits
        except Exception as e:
            p_ts.append(np.nan); elapsed_each.append(np.nan); failures.append({'video_id':row.video_id,'video_path':str(path),'error':repr(e)})
total_elapsed=time.perf_counter()-total_started
print('failures:',len(failures)); print('total sec:',total_elapsed); print('sec/video:',total_elapsed/len(wide))
if failures:
    display(pd.DataFrame(failures)); raise RuntimeError('model.ts inference failures detected')
wide['p_model_ts']=p_ts; wide['model_ts_elapsed_sec']=elapsed_each
print('[PASS] all 104 videos inferred')

In [ ]:
# 9. Sample-level comparison + summary
wide['abs_diff']=np.abs(wide['p_model_ts']-wide['p_expected'])
wide['signed_diff']=wide['p_model_ts']-wide['p_expected']
wide['model_ts_pred']=np.where(wide['p_model_ts']>=THRESHOLD,'RERECORDED','ORIGINAL')
wide['label_flip_vs_expected']=wide['model_ts_pred']!=wide['expected_pred']
wide['expected_margin_to_0.5']=np.abs(wide['p_expected']-THRESHOLD)
wide['model_ts_margin_to_0.5']=np.abs(wide['p_model_ts']-THRESHOLD)

diff=wide['abs_diff'].to_numpy(float)
summary={
 'num_videos':int(len(wide)),'alpha_06':float(ALPHA_06),'threshold':float(THRESHOLD),
 'a7_macro_f1_at_0.5':macro_f1(wide['label'],np.where(wide['p_a7']>=THRESHOLD,'RERECORDED','ORIGINAL')),
 'adapted_06_macro_f1_at_0.5':macro_f1(wide['label'],np.where(wide['p_06']>=THRESHOLD,'RERECORDED','ORIGINAL')),
 'expected_blend_macro_f1_at_0.5':macro_f1(wide['label'],wide['expected_pred']),
 'model_ts_macro_f1_at_0.5':macro_f1(wide['label'],wide['model_ts_pred']),
 'mean_abs_diff':float(np.mean(diff)),'median_abs_diff':float(np.median(diff)),
 'p90_abs_diff':float(np.quantile(diff,.90)),'p95_abs_diff':float(np.quantile(diff,.95)),
 'p99_abs_diff':float(np.quantile(diff,.99)),'max_abs_diff':float(np.max(diff)),
 'num_label_flips_vs_expected':int(wide['label_flip_vs_expected'].sum()),
 'total_inference_seconds':float(total_elapsed),'mean_inference_seconds':float(np.nanmean(wide['model_ts_elapsed_sec']))
}
print(json.dumps(summary,indent=2,ensure_ascii=False))

In [ ]:
# 10. Largest probability differences
largest=wide.sort_values('abs_diff',ascending=False)[[
 'video_id','label','p_a7','p_06','p_expected','p_model_ts','signed_diff','abs_diff',
 'expected_pred','model_ts_pred','label_flip_vs_expected'
]].head(20)
display(largest)

In [ ]:
# 11. Threshold flips
flips=wide.loc[wide['label_flip_vs_expected']].sort_values('expected_margin_to_0.5')[[
 'video_id','label','p_a7','p_06','p_expected','p_model_ts','abs_diff',
 'expected_pred','model_ts_pred','expected_margin_to_0.5'
]]
print('threshold flips:',len(flips)); display(flips)

In [ ]:
# 12. Classification correctness changes
wide['expected_correct']=wide['expected_pred']==wide['label']
wide['model_ts_correct']=wide['model_ts_pred']==wide['label']
error_change=wide.loc[wide['expected_correct']!=wide['model_ts_correct']][[
 'video_id','label','p_expected','p_model_ts','abs_diff','expected_pred','model_ts_pred','expected_correct','model_ts_correct'
]].sort_values('abs_diff',ascending=False)
print('samples whose correctness changed:',len(error_change)); display(error_change)

## Interpretation

- **label flip = 0이고 model.ts F1 == expected blend F1**: 제출 TorchScript/FP32가 DLC 의사결정을 망가뜨린 증거는 없다. public 하락은 synthetic-domain generalization 실패 쪽이 더 유력하다.
- **label flip이 있거나 DLC F1이 1.0에서 하락**: FP32/AMP 차이 또는 TorchScript export path가 실제 결정을 바꿨다. 먼저 export/inference를 조사한다.
- probability drift가 있어도 label flip이 0이면 DLC에는 영향이 없지만, hidden의 borderline sample에는 영향을 줄 가능성은 남는다.

In [ ]:
# 13. Automatic verdict
expected_f1=summary['expected_blend_macro_f1_at_0.5']; ts_f1=summary['model_ts_macro_f1_at_0.5']; n_flips=summary['num_label_flips_vs_expected']; max_diff=summary['max_abs_diff']
print('='*72); print('VERDICT'); print('='*72)
if n_flips==0 and abs(expected_f1-ts_f1)<1e-12:
    print('[PASS] model.ts has exactly the same threshold-0.5 decisions as the CSV blend on all 104 DLC videos.')
    print('=> Public-score drop is unlikely to be caused by a DLC-visible TorchScript/FP32 decision bug.')
    if max_diff>1e-3:
        print(f'[NOTE] max probability drift={max_diff:.6g}. No DLC label flip, but hidden borderline samples may still be sensitive.')
else:
    print('[ALERT] model.ts changes at least one DLC decision or Macro-F1.')
    print(f'expected F1={expected_f1:.12f}, model.ts F1={ts_f1:.12f}, flips={n_flips}')
    print('=> Investigate FP32 vs AMP / TorchScript export before blaming domain shift entirely.')
print('='*72)

In [ ]:
# 14. Save diagnostics next to model.ts
OUT_CSV=BLEND_DIR/'model_ts_vs_csv_dlc_predictions.csv'
OUT_JSON=BLEND_DIR/'model_ts_vs_csv_dlc_summary.json'
save_columns=[
 'video_id','label','video_path','p_a7','p_06','p_expected','p_model_ts','signed_diff','abs_diff',
 'expected_pred','model_ts_pred','label_flip_vs_expected','expected_correct','model_ts_correct',
 'expected_margin_to_0.5','model_ts_margin_to_0.5','model_ts_elapsed_sec'
]
wide[save_columns].to_csv(OUT_CSV,index=False)
with OUT_JSON.open('w',encoding='utf-8') as f: json.dump(summary,f,ensure_ascii=False,indent=2)
print('saved:',OUT_CSV); print('saved:',OUT_JSON)